# Bottle Vision — SAM 3 Colab Runner

This notebook prepares a clean Colab GPU runtime, installs the current official SAM 3 stack, runs Bottle Vision segmentation, and shows Original / Mask / Overlay for human review.

**Important:** use a fresh Colab GPU runtime for the first setup. SAM 3 currently requires Python 3.12+, PyTorch 2.7+, and a CUDA-compatible GPU; the official installation currently uses PyTorch 2.10.0 with CUDA 12.8 wheels.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/content/Bottlevision')

if REPO.exists():
    print(f'Repository already exists: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=False)
else:
    subprocess.run([
        'git', 'clone',
        'https://github.com/Rollerboy22/Bottlevision.git',
        str(REPO),
    ], check=True)

%cd /content/Bottlevision
print('Python:', sys.version)

## 1. Install a CUDA-compatible PyTorch stack

Do this **before importing torch**. The previous notebook allowed Colab's preinstalled PyTorch/CUDA combination to conflict with SAM 3. We now install the official current PyTorch CUDA 12.8 wheel explicitly.

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(f'SAM 3 now requires Python 3.12+. Colab is using {sys.version}. Start a current Python 3.12 runtime.')

%pip install -q --upgrade pip
%pip install -q --upgrade 'torch==2.10.0' torchvision --index-url https://download.pytorch.org/whl/cu128
%pip install -q -e .
%pip install -q -e '.[sam3]'
print('Installation finished. If Colab asks for a runtime restart, restart it and continue from the verification cell.')

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('Torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is not available. In Colab choose Runtime -> Change runtime type -> GPU, then restart and run this cell again.'
    )

print('GPU:', torch.cuda.get_device_name(0))
print('GPU capability:', torch.cuda.get_device_capability(0))

## 2. Load Bottle Vision

SAM 3 checkpoints require access approval on Hugging Face. If the model download later reports an authentication/access error, authenticate with a Hugging Face token that has access to the SAM 3 checkpoint.

In [ ]:
from colab.BottleVision_SAM3_Runner import (
    build_review_views,
    load_bottle_vision_config,
    load_rgb_image,
    run_segmentation,
    summarize_result,
)

config = load_bottle_vision_config()
print('Segmentation config:', config['segmentation'])

In [ ]:
from google.colab import files

uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
image = load_rgb_image(IMAGE_PATH)
result = run_segmentation(image, config)
summary = summarize_result(result)
summary

In [ ]:
import matplotlib.pyplot as plt

views = build_review_views(image, result, alpha=0.45)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for axis, (title, view) in zip(axes, views.items()):
    axis.imshow(view)
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()

## Human review checkpoint

Check **Original / Mask / Overlay**. A weak or incorrect mask must remain REVIEW/RESEGMENT rather than being silently accepted. Bounding boxes are never used as the final annotation.